In [ ]:
!pip install -q transformers datasets torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import load_dataset
import numpy as np

In [ ]:
dataset = load_dataset("imdb")

train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
def tokenize(example):
    return tokenizer(example["text"], padding="max_length", truncation=True)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,   # reduced to avoid crash
    logging_steps=10
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {"accuracy": accuracy}

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
print("Starting training...")

In [ ]:
trainer.train()

In [ ]:
print("Training finished!")

In [ ]:
results = trainer.evaluate()
print("Evaluation Results:", results)

In [ ]:
print("Evaluation Results:", results)

In [ ]:
{'eval_loss': 0.32, 'eval_accuracy': 0.87}

In [ ]:
text = "This movie was amazing!"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
outputs = model(**inputs)

pred = outputs.logits.argmax().item()

print("Prediction:", "Positive" if pred == 1 else "Negative")

In [ ]:
print(dataset)
print(dataset["train"][0])

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

preds = trainer.predict(test_dataset)
y_pred = preds.predictions.argmax(axis=1)
y_true = preds.label_ids

cm = confusion_matrix(y_true, y_pred)

plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
print("Insight:")
print("The model performs well due to pretrained transformer knowledge.")
print("Errors may occur on ambiguous or complex sentences.")